# Exercise: Autoregressive Stock Forecasting with a Transformer Decoder

## Part 1 — Data Download & Feature Engineering

Download daily OHLCV data for **at least 10 equities** and engineer a set of stationary features suitable for return forecasting. Define a target variable and split the data into train and test sets. Train a single shared model across all equities.

**Data sources:**
- `yfinance` (https://github.com/ranaroussi/yfinance) — free Python library wrapping Yahoo Finance; covers most global equities, ETFs, and indices
- Dukascopy (https://www.dukascopy.com/trading-tools/widgets/quotes/historical_data_feed) — free tick and daily OHLCV data for equities, forex, and commodities; downloadable as CSV
- Kaggle Datasets (https://www.kaggle.com/datasets) — search for 'stock prices'; many curated multi-ticker CSVs available for download
- Stooq (https://stooq.com/db/h/) — free historical daily data downloadable as CSV files, no API key required
- Alpha Vantage (https://www.alphavantage.co/) — free tier API (25 requests/day) with daily adjusted prices
- WRDS / CRSP (https://wrds-www.wharton.upenn.edu/) — institutional-grade source available through many university libraries

**Suggested features to engineer:**

*Returns & volatility*
- Daily log-return
- Rolling realised volatility (std of log-returns over 5 and 20 days)
- Overnight gap (log of Open / previous Close)
- Intraday range normalised by Close: $(High - Low) / Close$

*Trend indicators*
- Simple Moving Average crossover: SMA(5) / SMA(20) − 1
- Exponential Moving Average crossover: EMA(12) / EMA(26) − 1
- MACD signal line: EMA(12) − EMA(26) normalised by price

*Momentum indicators*
- Rate of Change (ROC): log-return over 5, 10, and 20 days
- Relative Strength Index (RSI, 14-day)

*Volume indicators*
- Volume log-change
- On-Balance Volume (OBV) normalised by its rolling mean

*Mean-reversion indicators*
- Bollinger Band position: $(Close - SMA(20)) / (2 \times \sigma_{20})$
- Distance from 52-week high/low

You are not required to use all of these — select and justify the subset you include.

## Part 2 — Transformer Decoder Architecture

Implement a decoder-only transformer for next-step return prediction. Your model must enforce causal (masked) self-attention so that position $t$ cannot attend to any future position.

**Q1.** Print the causal mask for a sequence of length 5. Why are the diagonal entries $0$ and not $-\infty$? What would happen if you removed the mask entirely?

*Your answer here.*

## Part 3 — Training

Build a sliding-window dataset and train your model. Plot the training and validation loss curves.

## Part 4 — Evaluation

Evaluate the model on the test set. Report test MSE and directional accuracy. Compare against an LSTM baseline with the same architecture depth.

## Part 5 — Attention Inspection

Visualise the attention weight matrices as heatmaps for at least two layers and two heads. Compare a high-volatility window with a calm one.

**Q2.** What temporal patterns, if any, do the attention maps reveal? Do different layers focus on different lags?

*Your answer here.*

## Part 6 — Trading Strategy

Convert the model's predictions into a long/short/flat signal using a threshold $\tau$. Compute gross and net daily P&L, applying a transaction cost of 5 basis points on every position change.

**Q3.** Sweep $\tau$ over several values and plot Sharpe ratio vs $\tau$. What is the trade-off between a high and a low threshold? Does transaction cost affect the optimal $\tau$?

*Your answer here.*

## Part 7 — Risk Metrics

Implement a function `compute_risk_metrics(returns)` that takes a daily return series and returns the following metrics (annualised where appropriate, using 252 trading days per year). Apply it to the strategy (gross and net) and a buy-and-hold benchmark, and display the results in a single comparison table.

| Metric | Formula |
|--------|---------|
| Annualised Return | $\mu \times 252$ |
| Annualised Volatility | $\sigma \times \sqrt{252}$ |
| Total Return | $\exp(\sum r_t) - 1$ |
| Sharpe Ratio | Ann. return / Ann. volatility |
| Sortino Ratio | Ann. return / downside volatility |
| Maximum Drawdown | $\min_t \frac{W_t - \max_{s \leq t} W_s}{\max_{s \leq t} W_s}$ |
| Calmar Ratio | Ann. return / |Max Drawdown| |
| VaR (95%) | 5th percentile of daily returns |
| CVaR (95%) | Mean of returns below VaR |
| Win Rate | Fraction of days with positive return |
| Profit Factor | Mean win / |Mean loss| |

## Part 8 — Tear Sheet

Produce a four-panel figure: cumulative returns vs buy-and-hold, rolling 30-day Sharpe, drawdown series, and daily return distribution with VaR and CVaR marked.

## Part 9 — Discussion

**Q4.** The Sharpe ratio assumes normally distributed returns. How does this limit its reliability for financial strategies? Which metric in your table is more robust to extreme losses?

*Your answer here.*

**Q5.** Why is a simple train/test split insufficient for evaluating a trading strategy? What does walk-forward validation address that a single split does not?

*Your answer here.*

**Q6.** Compare CVaR between your net strategy and buy-and-hold. Under what conditions would a lower-Sharpe, lower-CVaR strategy be preferable?

*Your answer here.*

**Q7.** If directional accuracy is only marginally above 50%, does that mean the model is useless? Suggest two alternative applications in quantitative finance where the model could still add value.

*Your answer here.*